# 🔬 Energy Benchmark — T4 GPU
**Third hardware point for cross-hardware comparison**

Run all cells in order. Results auto-save to Google Drive.
Expected runtime: ~30-40 min on T4.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers==4.47.1 diffusers accelerate pynvml audiocraft scipy
import torch
assert torch.cuda.is_available(), 'No GPU!'
gpu_name = torch.cuda.get_device_properties(0).name
print(f'GPU: {gpu_name}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Cell 2: Setup
import time, json, os, gc
import numpy as np
import pynvml
from threading import Thread, Event
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/project_energy_results'
os.makedirs(SAVE_DIR, exist_ok=True)

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
GPU_NAME = pynvml.nvmlDeviceGetName(handle)
if isinstance(GPU_NAME, bytes): GPU_NAME = GPU_NAME.decode()
BACKEND = 'cuda'
REPEATS = 30
print(f'Hardware: {GPU_NAME}, Backend: {BACKEND}, Repeats: {REPEATS}')

In [ ]:
# Cell 3: Power monitor
class PowerMonitor:
    def __init__(self):
        self.samples = []
        self._stop = Event()
    def _record(self):
        while not self._stop.is_set():
            try:
                p = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
                self.samples.append((time.time(), p))
            except: pass
            time.sleep(0.1)
    def start(self):
        self.samples = []
        self._stop.clear()
        self._thread = Thread(target=self._record, daemon=True)
        self._thread.start()
    def stop(self):
        self._stop.set()
        self._thread.join()
        if len(self.samples) < 2: return {}
        times = [s[0] for s in self.samples]
        powers = [s[1] for s in self.samples]
        dt = [times[i+1]-times[i] for i in range(len(times)-1)]
        energy = sum((powers[i]+powers[i+1])/2*dt[i] for i in range(len(dt)))
        return {'total_energy_j': round(energy,2), 'avg_power_w': round(np.mean(powers),2),
                'max_power_w': round(max(powers),2), 'duration_sec': round(times[-1]-times[0],2),
                'samples': len(self.samples)}
print('PowerMonitor ready')

In [ ]:
# Cell 4: TEXT experiment
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = 'microsoft/Phi-3-mini-4k-instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16,
    trust_remote_code=True, device_map='cuda')
prompt = 'Write a short story about a robot learning to paint.'
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

text_results = []
for max_tok in [64, 128, 256, 512]:
    print(f'  Text max_tokens={max_tok}')
    _ = model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)  # warmup
    for run in range(1, REPEATS+1):
        torch.cuda.synchronize()
        pm = PowerMonitor(); pm.start()
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)
        torch.cuda.synchronize()
        gen_time = time.time() - t0
        stats = pm.stop()
        stats.update({'generation_time_sec': round(gen_time,2), 'max_tokens': max_tok,
                      'modality': 'text', 'model': 'Phi-3-mini-4k', 'params_B': 3.8,
                      'run': run, 'hardware': GPU_NAME, 'backend': BACKEND})
        text_results.append(stats)

gpu_tag = GPU_NAME.replace(' ','-')
with open(f'{SAVE_DIR}/exp1_text_{gpu_tag}.json','w') as f: json.dump(text_results,f,indent=2)
del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
print(f'✅ Text done: {len(text_results)} runs')

In [ ]:
# Cell 5: IMAGE experiment
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16).to('cuda')
prompt = 'A serene landscape with mountains and a lake at sunset, highly detailed, 8k'

image_results = []
for res in [256, 384, 512]:
    for steps in [20, 30]:
        print(f'  Image {res}x{res}, {steps} steps')
        _ = pipe(prompt, height=res, width=res, num_inference_steps=steps,
                guidance_scale=7.5, generator=torch.Generator('cuda').manual_seed(0))
        for run in range(1, REPEATS+1):
            torch.cuda.synchronize()
            pm = PowerMonitor(); pm.start()
            t0 = time.time()
            _ = pipe(prompt, height=res, width=res, num_inference_steps=steps,
                    guidance_scale=7.5, generator=torch.Generator('cuda').manual_seed(run))
            torch.cuda.synchronize()
            gen_time = time.time() - t0
            stats = pm.stop()
            stats.update({'generation_time_sec': round(gen_time,2), 'resolution': f'{res}x{res}',
                          'steps': steps, 'modality': 'image', 'model': 'SD-v1-5', 'params_B': 0.9,
                          'run': run, 'hardware': GPU_NAME, 'backend': BACKEND})
            image_results.append(stats)

with open(f'{SAVE_DIR}/exp2_image_{gpu_tag}.json','w') as f: json.dump(image_results,f,indent=2)
del pipe; gc.collect(); torch.cuda.empty_cache()
print(f'✅ Image done: {len(image_results)} runs')

In [ ]:
# Cell 6: VIDEO experiment
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
adapter = MotionAdapter.from_pretrained('guoyww/animatediff-motion-adapter-v1-5-3',
    torch_dtype=torch.float16)
vpipe = AnimateDiffPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5',
    motion_adapter=adapter, torch_dtype=torch.float16).to('cuda')
vpipe.scheduler = DDIMScheduler.from_config(vpipe.scheduler.config,
    beta_schedule='linear', clip_sample=False, timestep_spacing='linspace', steps_offset=1)
prompt = 'A serene landscape with mountains and a lake at sunset, highly detailed, 8k'

video_results = []
for frames in [4, 8, 12]:
    for steps in [20, 30]:
        print(f'  Video {frames}f, {steps} steps, 256x256')
        _ = vpipe(prompt, num_frames=frames, height=256, width=256,
                 num_inference_steps=steps, generator=torch.Generator('cuda').manual_seed(0))
        for run in range(1, REPEATS+1):
            torch.cuda.synchronize()
            pm = PowerMonitor(); pm.start()
            t0 = time.time()
            _ = vpipe(prompt, num_frames=frames, height=256, width=256,
                     num_inference_steps=steps, generator=torch.Generator('cuda').manual_seed(run))
            torch.cuda.synchronize()
            gen_time = time.time() - t0
            stats = pm.stop()
            stats.update({'generation_time_sec': round(gen_time,2), 'frames': frames,
                          'steps': steps, 'resolution': '256x256', 'modality': 'video',
                          'model': 'AnimateDiff-v1.5', 'run': run,
                          'hardware': GPU_NAME, 'backend': BACKEND})
            video_results.append(stats)

with open(f'{SAVE_DIR}/exp3_video_{gpu_tag}.json','w') as f: json.dump(video_results,f,indent=2)
del vpipe, adapter; gc.collect(); torch.cuda.empty_cache()
print(f'✅ Video done: {len(video_results)} runs')

In [ ]:
# Cell 7: MUSIC experiment
from audiocraft.models import MusicGen

music_results = []
for model_name, params_m in [('small', 589), ('medium', 2008)]:
    mg = MusicGen.get_pretrained(f'facebook/musicgen-{model_name}')
    mg.set_generation_params(use_sampling=True, top_k=250)
    desc = ['happy rock song with electric guitar and drums']
    tokens_list = [128, 256, 512] if model_name == 'small' else [128, 256]
    for max_tok in tokens_list:
        mg.set_generation_params(use_sampling=True, top_k=250, max_tokens=max_tok)
        audio_sec = round(max_tok / 50, 1)
        print(f'  Music {model_name} {max_tok}tok ({audio_sec}s)')
        _ = mg.generate(desc)
        for run in range(1, REPEATS+1):
            torch.cuda.synchronize()
            pm = PowerMonitor(); pm.start()
            t0 = time.time()
            wav = mg.generate(desc)
            torch.cuda.synchronize()
            gen_time = time.time() - t0
            stats = pm.stop()
            stats.update({'generation_time_sec': round(gen_time,2), 'audio_sec': audio_sec,
                          'max_tokens': max_tok, 'modality': 'music',
                          'model': f'MusicGen-{model_name}', 'params_M': params_m,
                          'run': run, 'hardware': GPU_NAME, 'backend': BACKEND})
            music_results.append(stats)
    del mg; gc.collect(); torch.cuda.empty_cache()

with open(f'{SAVE_DIR}/exp4_music_{gpu_tag}.json','w') as f: json.dump(music_results,f,indent=2)
print(f'✅ Music done: {len(music_results)} runs')

In [ ]:
# Cell 8: Summary
import datetime
summary = {
    'hardware': GPU_NAME, 'backend': BACKEND, 'repeats': REPEATS,
    'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'pytorch': torch.__version__, 'cuda': torch.version.cuda,
    'text': {'n_records': len(text_results),
             'energy_range_j': [round(min(r['total_energy_j'] for r in text_results),1),
                                round(max(r['total_energy_j'] for r in text_results),1)]},
    'image': {'n_records': len(image_results),
              'energy_range_j': [round(min(r['total_energy_j'] for r in image_results),1),
                                 round(max(r['total_energy_j'] for r in image_results),1)]},
    'video': {'n_records': len(video_results),
              'energy_range_j': [round(min(r['total_energy_j'] for r in video_results),1),
                                 round(max(r['total_energy_j'] for r in video_results),1)]},
    'music': {'n_records': len(music_results),
              'energy_range_j': [round(min(r['total_energy_j'] for r in music_results),1),
                                 round(max(r['total_energy_j'] for r in music_results),1)]},
}
with open(f'{SAVE_DIR}/summary_{gpu_tag}.json','w') as f: json.dump(summary,f,indent=2)
print('\n🎉 ALL EXPERIMENTS COMPLETE!')
print(json.dumps(summary, indent=2))